# 01 · Discovery: Nodes, Metrics, and Lifetimes

**Goal: learn what is out there before you pull data.**

In notebook `00` you queried temperature from "anywhere in the network." That is
fine for a first probe, but real work starts with specific questions about
specific nodes: *Which nodes are reporting right now? What does this particular
node measure? How long has it been running?* This notebook answers those three
discovery questions.

Discovery matters because the alternative is guessing. You cannot analyze a node
whose call sign you do not know, and you cannot plot a metric you did not know
existed. A few minutes of discovery up front saves you from queries that quietly
return nothing because you filtered on a name the node never reports.

In [ ]:
import sage_data_client
import pandas as pd

In [ ]:
# Import the SageTools helper package (github.com/mpapka/SageTools).
#
# SageTools is a convenience layer built on top of the official sage_data_client.
# It is optional. Every lab in this course also shows the underlying
# sage_data_client call directly, so the helper is never a hard dependency.
#
# The try/except means the notebook runs whether or not the package is installed:
# if it is missing, haveSageTools becomes False and the notebook falls back to
# the direct calls it already teaches.
try:
    import sage
    haveSageTools = True
except ImportError:
    haveSageTools = False

print("SageTools helper package available:", haveSageTools)

## 1. The one primitive: `sage_data_client.query`

Every discovery technique in this notebook is the same function call with
different arguments. Here is the full signature, with each argument explained.
You do not need all of them at once; most queries use two or three.

```python
sage_data_client.query(
    start,          # REQUIRED. Where the time window begins.
    end=None,       # Optional. Where it ends. Defaults to now.
    head=None,      # Optional. Return only the first N matching rows.
    tail=None,      # Optional. Return only the last N matching rows.
    filter=None,    # Optional. A dict that narrows what comes back.
)
```

- **`start` and `end`** define the time window. They accept either a relative
  offset from now (`-5m`, `-1h`, `-7d`) or an absolute UTC timestamp
  (`2024-06-01T00:00:00Z`). Notebook `02` covers the grammar in full. For
  discovery, short relative windows are almost always what you want.
- **`head` and `tail`** limit how many rows return, from the start or the end of
  the window respectively. They make a query cheap, because the Beehive does not
  have to stream the entire window back to you.
- **`filter`** is a dictionary. Each key narrows the result. The keys you will
  use most in discovery:
  - `vsn`: restrict to one node.
  - `name`: restrict to one metric.
  - `plugin`: restrict to measurements produced by a particular plugin.

A query with a `vsn` filter but no `name` filter returns every metric that node
reported in the window. That single fact is the engine of metric discovery,
which is the next section.

## 2. What does this node measure? Listing metrics

To find every quantity a node reports, query it by `vsn` with no `name` filter,
then look at the unique values in the `name` column. Keep the time window short,
a few minutes, so the query stays fast and only surfaces metrics that are
currently active.

The function below does exactly that and prints the result. Read it line by line:
it queries, checks for an empty result, pulls the unique metric names, sorts
them, and prints them.

In [ ]:
def discoverMetrics(nodeVsn, timeRange="-10m"):
    """
    List every metric a node reported within a recent time window.

    Args:
        nodeVsn: the node call sign to inspect, for example "W06C".
        timeRange: how far back to look, as a relative offset. Default 10 minutes.

    Returns:
        A sorted list of metric name strings. Empty if the node reported nothing
        in the window, which usually means it is offline or the window is short.
    """
    recentDf = sage_data_client.query(
        start=timeRange,
        filter={"vsn": nodeVsn},   # note: no "name" key, so all metrics return
    )

    if recentDf.empty:
        print(f"No data from {nodeVsn} in the last {timeRange}.")
        print("The node may be offline, or try a longer window like '-1h'.")
        return []

    metricNames = sorted(recentDf["name"].unique().tolist())
    print(f"{nodeVsn} reported {len(metricNames)} distinct metrics in {timeRange}:")
    for metricName in metricNames:
        print("  -", metricName)
    return metricNames


# Example. If this node is quiet, swap in one you find live in section 5.
metricNames = discoverMetrics("W06C")

### Reading the output

You will typically see two families of names:

- **`sys.*`** metrics are about the node itself: `sys.uptime` (seconds since the
  node last booted), `sys.mem.*`, `sys.cpu.*`, `sys.gps.*` (location). These come
  from the node's own operating system and are a reliable sign of life.
- **`env.*`** and similar metrics are the science payload: `env.temperature`,
  `env.relative_humidity`, `env.pressure`, and others depending on which sensors
  the node carries.

Two different nodes can report different metric sets, because they carry
different hardware. That is exactly why you discover before you query.

The same capability ships in SageTools as `listMetrics`. It does what
`discoverMetrics` does, so use whichever you prefer.

In [ ]:
if haveSageTools:
    from sage.beehive.discovery.listMetrics import listMetrics
    listMetrics("W06C", timeRange="-10m")
else:
    print("SageTools not installed. Use discoverMetrics above, it does the same thing.")

## 3. `head` and `tail`: probing the edges cheaply

`head=N` returns the first N rows of a window; `tail=N` returns the last N. These
are the cheapest queries you can run, because the Beehive answers them without
streaming the whole range.

The mental model:

- `tail=1` answers "is this node alive, and what did it just say?" You point it at
  the recent past and ask for the newest record.
- `head=1` answers "when did this stream begin?" You point it at a wide window and
  ask for the oldest record.

`sys.uptime` is the metric of choice for liveness checks. Every healthy node
emits it on a regular heartbeat, and it is cheap to query, so it is the standard
probe for "is this node up."

In [ ]:
# Liveness check: the single most recent heartbeat from a node.
lastBeatDf = sage_data_client.query(
    start="-7d",                                   # look back up to a week
    tail=1,                                         # only the newest record
    filter={"vsn": "W06C", "name": "sys.uptime"},  # heartbeat metric
)

if lastBeatDf.empty:
    print("No heartbeat in the last 7 days. This node is likely offline.")
else:
    beatTime = pd.to_datetime(lastBeatDf["timestamp"].iloc[0])
    uptimeSeconds = float(lastBeatDf["value"].iloc[0])
    print("last heartbeat at:", beatTime, "UTC")
    print("node uptime      :", round(uptimeSeconds / 86400, 1), "days since last boot")

## 4. How long has this node been running? Finding a lifetime

To find when a node first appeared, you cannot simply scan its entire history in
one query. A query that spans years of data can exceed the Beehive's time limit
and fail with a server error. The robust pattern probes **one year at a time**
with `head=1` until it finds the first record, then takes the newest record with
`tail=1`.

The function below implements that pattern. Walk through the logic:

1. Find the last record. Look at the recent past first, since that is fast, and
   only widen to the full history if nothing recent turns up.
2. Find the first record by probing each year from an early start year forward.
   The first year that returns any data contains the node's birth.
3. Report first seen, last seen, and the span between them.

This is a compact version of the SageTools `findNodeLife` tool, written so you
can read the whole thing.

In [ ]:
from datetime import datetime


def findLifetime(nodeVsn, firstYear=2019):
    """
    Find the first and last sys.uptime record for a node.

    Args:
        nodeVsn: node call sign, for example "W06C".
        firstYear: earliest year to probe for the node's birth. Sage data begins
            around 2019, so that is a sensible floor.

    Returns:
        A (firstSeen, lastSeen) tuple of pandas Timestamps, or None if the node
        has no sys.uptime records at all.
    """
    # Step 1: the last record. Try the recent past first (fast), then widen.
    lastDf = sage_data_client.query(
        start="-30d", tail=1,
        filter={"vsn": nodeVsn, "name": "sys.uptime"},
    )
    if lastDf.empty:
        lastDf = sage_data_client.query(
            start="2018-01-01T00:00:00Z", tail=1,
            filter={"vsn": nodeVsn, "name": "sys.uptime"},
        )
    if lastDf.empty:
        print(f"No sys.uptime records for {nodeVsn}. Check the call sign.")
        return None
    lastSeen = pd.to_datetime(lastDf["timestamp"].iloc[0])

    # Step 2: the first record. Probe year by year until data appears.
    firstSeen = None
    for year in range(firstYear, datetime.now().year + 1):
        probeDf = sage_data_client.query(
            start=f"{year}-01-01T00:00:00Z",
            end=f"{year + 1}-01-01T00:00:00Z",
            head=1,
            filter={"vsn": nodeVsn, "name": "sys.uptime"},
        )
        if not probeDf.empty:
            firstSeen = pd.to_datetime(probeDf["timestamp"].iloc[0])
            break

    if firstSeen is None:
        print(f"Could not pinpoint a start year for {nodeVsn}.")
        return None

    # Step 3: report.
    print(f"Node {nodeVsn}")
    print(f"  first seen: {firstSeen}")
    print(f"  last seen:  {lastSeen}")
    print(f"  lifetime:   {lastSeen - firstSeen}")
    return firstSeen, lastSeen


lifetime = findLifetime("W06C")

In [ ]:
# The SageTools equivalent, if installed. It adds a rich formatted report and
# suggests a batch extraction command for the node's full history.
if haveSageTools:
    from sage.beehive.discovery.nodeLife import findNodeLife
    findNodeLife("W06C")

## 5. Finding nodes that are live right now

The examples above use fixed call signs, but the set of online nodes changes
over time. To find nodes that are actually reporting at this moment, query a
common heartbeat metric across the whole network over a short window and list the
distinct nodes that answered.

In [ ]:
def findLiveNodes(timeRange="-10m"):
    """Return the sorted list of node VSNs reporting sys.uptime in the window."""
    liveDf = sage_data_client.query(
        start=timeRange,
        filter={"name": "sys.uptime"},
    )
    if liveDf.empty:
        print("No nodes reported in the window. Try a longer range like '-1h'.")
        return []
    liveVsns = sorted(liveDf["meta.vsn"].dropna().unique().tolist())
    print(f"{len(liveVsns)} nodes reported in the last {timeRange}:")
    print(", ".join(liveVsns))
    return liveVsns


liveVsns = findLiveNodes("-10m")

### Putting discovery together

A natural workflow chains the three techniques. Pick a live node, list what it
measures, and check how long it has been running. The cell below does that
automatically using the first live node found above, so it adapts to whatever is
online when you run it.

In [ ]:
if liveVsns:
    chosenVsn = liveVsns[0]
    print(f"Inspecting the first live node: {chosenVsn}\n")
    discoverMetrics(chosenVsn, timeRange="-10m")
    print()
    findLifetime(chosenVsn)
else:
    print("No live nodes found. Widen the window in findLiveNodes and retry.")

## 6. Exercises

1. Run `findLiveNodes` with `-1h` instead of `-10m`. Do more nodes appear, and
   why would a longer window surface additional nodes?
2. Pick two live nodes and call `discoverMetrics` on each. Convert each result to
   a `set` and compute the intersection. Which metrics do both nodes share, and
   which are unique to one?
   *Hint: `set(listA) & set(listB)` gives the shared items.*
3. Use `findLifetime` on three different nodes. Which has been running longest?
   State the measured lifetimes first, then, in a separate sentence, offer one
   guess about why a node might have a short lifetime. Keep the fact and the guess
   apart.
4. A node returns nothing for `tail=1` over the last 7 days but does return a row
   for `head=1` in 2022. What can you conclude about the node's current status,
   and what does the 2022 result tell you that the empty recent query does not?
5. Modify `findLiveNodes` to also report, for each live node, how many rows it
   contributed in the window. Which node is the chattiest?
   *Hint: a groupby on `meta.vsn` with `.size()` gives per node counts.*

## Further reading

- Sage query filters and time formats: https://sagecontinuum.org/docs/reference-guides/dev-quick-reference
- Browse a node's deployment in the portal: https://portal.sagecontinuum.org/nodes/W06C
- The official data client: https://github.com/sagecontinuum/sage-data-client
- pandas `unique` and `groupby` for summarizing results: https://pandas.pydata.org/docs/